# Зависимости

In [29]:
%pip install gigachat openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Запросы

Каждой модели отправим 4 запроса:

1. Python
 

    Напиши функцию на Python, которая принимает строку и возвращает словарь,
    где ключи — слова, а значения — сколько раз каждое слово встречается в строке.
    Регистр не учитывать, знаки препинания игнорировать.
    Приведи код, кратко объясни решение и покажи пример вызова с результатом.

 
2. C
 

    Напиши программу на C, которая считывает с клавиатуры массив из n целых чисел
    (n вводится первым), сортирует его пузырьковой сортировкой и выводит результат.
    Память выдели динамически и не забудь её освободить.
    Приведи полный компилируемый код и кратко поясни, как он работает.

 
3. Математика
 

    Дай строгое определение математического ожидания случайной величины
    для дискретного и непрерывного случаев.
    Перечисли основные свойства математического ожидания (линейность, ожидание константы,
    ожидание произведения независимых величин и т.д.) с краткими доказательствами.
    Приведи по одному примеру вычисления для дискретной и непрерывной величины.

 
4. Литература
 

    Опиши главных действующих лиц рассказа И. С. Тургенева «Муму».
    Для каждого персонажа укажи: кто он, черты характера, роль в сюжете.
    Отдельно объясни, в чём смысл образа Герасима и какую идею хотел донести автор.


## Список промптов

In [11]:
prompts = [
    {
        "name": "python",
        "prompt": (
            "Напиши функцию на Python, которая принимает строку и возвращает словарь, "
            "где ключи — слова, а значения — сколько раз каждое слово встречается в строке. "
            "Регистр не учитывать, знаки препинания игнорировать. "
            "Приведи код, кратко объясни решение и покажи пример вызова с результатом."
        ),
    },
    {
        "name": "c",
        "prompt": (
            "Напиши программу на C, которая считывает с клавиатуры массив из n целых чисел "
            "(n вводится первым), сортирует его пузырьковой сортировкой и выводит результат. "
            "Память выдели динамически и не забудь её освободить. "
            "Приведи полный компилируемый код и кратко поясни, как он работает."
        ),
    },
    {
        "name": "math",
        "prompt": (
            "Дай строгое определение математического ожидания случайной величины "
            "для дискретного и непрерывного случаев. "
            "Перечисли основные свойства математического ожидания (линейность, ожидание константы, "
            "ожидание произведения независимых величин и т.д.) с краткими доказательствами. "
            "Приведи по одному примеру вычисления для дискретной и непрерывной величины."
        ),
    },
    {
        "name": "literature",
        "prompt": (
            "Опиши главных действующих лиц рассказа И. С. Тургенева «Муму». "
            "Для каждого персонажа укажи: кто он, черты характера, роль в сюжете. "
            "Отдельно объясни, в чём смысл образа Герасима и какую идею хотел донести автор."
        ),
    },
]

# Работа с GigaChat через API

Работу с GigaChat будем проводить через SDK

## Получение API ключа

Авторизируемся на https://developers.sber.ru и там получаем `Authorization key`.

Бесплатно доступно несколько десятков миллионов токенов.

Для работы с API нужно иметь сертификаты Минцифры. Скачаем их.

`mkdir -p ~/certs`

`wget https://gu-st.ru/content/lending/russian_trusted_root_ca_pem.crt -O ~/certs/russian_trusted_root_ca.crt`

In [26]:
import os
from datetime import datetime

from dotenv import load_dotenv
from gigachat import GigaChat

load_dotenv()

client = GigaChat(
    credentials=os.getenv("GIGACHAT_AUTH_KEY"),
    scope="GIGACHAT_API_PERS", # Для физ. лиц
    ca_bundle_file=os.path.expanduser("~/certs/russian_trusted_root_ca.crt"),
)

token = client.get_token()

dt = datetime.fromtimestamp(token.expires_at / 1000)
print(f"Ключ действителен до: {dt.strftime("%d.%m.%Y %H:%M:%S")}")

Ключ действителен до: 23.09.2026 01:18:06


## Получение списка доступных моделей

In [ ]:
models = client.get_models()
for model in models.data:
    print(f"{model.id_}")

response = client.chat.create("Привет, как дела?")

GigaChat-2
GigaChat-2-Max
GigaChat-2-Pro
GigaChat-3-Lightning
GigaChat-3-Pro
GigaChat-3-Ultra
Embeddings
Embeddings-2
EmbeddingsGigaR
GigaEmbeddings-3B-2025-09


## Отправка запросов на API

In [32]:
from gigachat.models import ChatCompletionRequest, ChatMessage

req = ChatCompletionRequest(
    model="GigaChat-2",
    messages=[ChatMessage(role="user", content="Привет, как дела?")],
)

answer = client.chat.create(req).messages[0].content[0].text
print(answer)

Привет! Всё отлично, заряжен на интересную беседу. А у тебя как дела?


При помощи роли `system` можно скорректировать поведение модели.

In [27]:
req = ChatCompletionRequest(
    model="GigaChat-2",
    messages=[
        ChatMessage(role="system", content="Отвечай одним предложением, в конце добавляй смайлик"),
        ChatMessage(role="user", content="Привет, как дела?")
    ],
)

answer = client.chat.create(req).messages[0].content[0].text
print(answer)

Привет, у меня всё отлично, спасибо! А у тебя как? 😊


### Отправим несколько запросов моделям GigaChat-3-Pro и GigaChat-3-Lightning

In [30]:
from IPython.display import Markdown, display

models = ["GigaChat-3-Pro", "GigaChat-3-Lightning"]

for model in models:
    display(Markdown(f"# {model}"))
    for prompt in prompts:
        req = ChatCompletionRequest(
            model=model,
            messages=[ChatMessage(role="user", content=prompt["prompt"])],
        )
        answer = client.chat.create(req).messages[0].content[0].text
        display(Markdown(f"## {prompt["name"]}\n\n{answer}\n\n---"))

# GigaChat-3-Pro

## python

## Решение задачи

Для подсчёта частоты слов в строке с игнорированием регистра и знаков препинания можно использовать следующий подход:

1. Привести всю строку к нижнему регистру, чтобы регистр не влиял на результат.
2. Удалить все знаки препинания (или заменить их пробелами), используя регулярные выражения или встроенные методы Python.
3. Разбить строку по пробельным символам для получения списка слов.
4. Пройти по списку слов и посчитать частоту каждого слова с помощью словаря.
5. Вернуть полученный словарь.

## Код функции

```python
import re
from collections import Counter

def word_frequency(text):
    # 1. Преобразуем текст в нижний регистр
    text = text.lower()
    
    # 2. Убираем знаки препинания (заменяем их пробелом)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 3. Делим текст на слова по пробелам
    words = text.split()
    
    # 4. Считаем частоту слов
    frequency_dict = dict(Counter(words))
    
    return frequency_dict
```

### Краткое объяснение решения

- **Регулярные выражения (`re`)** используются для удаления всех символов, кроме букв, цифр и пробелов.
- Метод `split()` разбивает строку на список слов по пробелам.
- Класс `Counter` из модуля `collections` автоматически считает количество элементов в списке.
- Результат преобразуется обратно в обычный словарь перед возвратом.

---

## Пример вызова функции

```python
text = "Привет! Как дела? Привет-привет, это тест. Тест, тест!"
result = word_frequency(text)
print(result)
```

### Вывод программы

```
{'привет': 3, 'как': 1, 'дела': 1, 'это': 1, 'тест': 3}
```

В этом примере:
- Все слова приведены к нижнему регистру.
- Знаки препинания удалены.
- Подсчёт выполнен корректно: слово «привет» встречается 3 раза, «тест» — тоже 3 раза, остальные слова — по одному разу.

---

## c

Вот полный компилируемый код на C, который считывает массив целых чисел с клавиатуры, сортирует его пузырьковой сортировкой и выводит результат:

```c
#include <stdio.h>
#include <stdlib.h>

// Функция для обмена двух элементов массива
void swap(int *a, int *b) {
    int temp = *a;
    *a = *b;
    *b = temp;
}

// Реализация пузырьковой сортировки
void bubbleSort(int *arr, int n) {
    for (int i = 0; i < n - 1; ++i) {
        // Флаг для оптимизации: если за проход не было обменов — массив уже отсортирован
        int swapped = 0;
        
        for (int j = 0; j < n - i - 1; ++j) {
            if (arr[j] > arr[j + 1]) {
                swap(&arr[j], &arr[j + 1]);
                swapped = 1;
            }
        }
        
        // Если за весь внутренний цикл ни одного обмена не произошло,
        // значит массив полностью отсортирован.
        if (!swapped)
            break;
    }
}

int main() {
    int n;
    
    // Ввод размера массива
    printf("Введите количество элементов в массиве: ");
    scanf("%d", &n);

    // Динамическое выделение памяти под массив
    int *arr = (int *)malloc(n * sizeof(int));
    if (arr == NULL) {
        fprintf(stderr, "Ошибка выделения памяти\n");
        return 1;
    }

    // Чтение элементов массива с клавиатуры
    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; ++i) {
        scanf("%d", &arr[i]);
    }

    // Сортировка массива
    bubbleSort(arr, n);

    // Вывод отсортированного массива
    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; ++i) {
        printf("%d ", arr[i]);
    }
    printf("\n");

    // Освобождение динамически выделенной памяти
    free(arr);

    return 0;
}
```

### Как работает программа

1. **Чтение данных**
   - Сначала запрашивается размер массива `n`.
   - Затем выделяется память под массив из $n$ элементов с помощью функции `malloc`. Проверяется успешность выделения памяти.
   - Далее вводятся сами элементы массива через стандартный ввод.

2. **Пузырьковая сортировка**
   - Алгоритм проходит по массиву несколько раз.
   - На каждом шаге сравниваются соседние элементы, и при необходимости они меняются местами (`swap`).
   - После каждого прохода самый большой элемент оказывается в конце массива ("всплывает" вверх), поэтому следующий проход можно делать короче.
   - Используется оптимизация: если во внутреннем цикле не было сделано ни одной перестановки, то массив уже отсортирован, и алгоритм завершает работу досрочно.

3. **Освобождение памяти**
   - По завершении работы вызывается функция `free`, чтобы освободить ранее выделенную память и избежать утечек.

Программа корректно обрабатывает ввод пользователя и освобождает ресурсы после завершения своей работы.

---

## math

## 1. Строгое определение математического ожидания

### Дискретный случай
Пусть $X$ — дискретная случайная величина, принимающая значения $x_1, x_2, \dots, x_n,\dots$ с вероятностями $p_i = P(X=x_i)$, где $\sum_{i} p_i = 1$. Тогда **математическое ожидание** (или среднее значение) определяется как сумма произведений значений на их вероятности:
$$
\mathbb{E}[X] = \sum_{i=1}^{\infty} x_i\,P(X = x_i)
$$
при условии абсолютной сходимости ряда.

### Непрерывный случай
Пусть $X$ — непрерывная случайная величина с плотностью распределения $f_X(x)$. Математическое ожидание определяется через интеграл по всей области определения:
$$
\mathbb{E}[X] = \int_{-\infty}^{+\infty} x\, f_X(x)\, dx,
$$
если интеграл сходится абсолютно.

---

## 2. Основные свойства математического ожидания

### 1. Ожидание константы
Для любой константы $c$:
$$
\mathbb{E}[c] = c
$$
**Доказательство:**
- В дискретном случае: единственная точка $c$ с вероятностью $1$, значит $\mathbb{E}[c]=c\cdot 1=c$.
- В непрерывном случае: плотность сосредоточена в точке $c$, интеграл даёт $c$.

### 2. Линейность (аддитивность и однородность)
Для любых случайных величин $X$ и $Y$ и констант $a,b$:
$$
\mathbb{E}[aX + bY] = a\,\mathbb{E}[X] + b\,\mathbb{E}[Y]
$$
**Доказательство для дискретного случая:**  
Рассмотрим сумму двух независимых или зависимых величин. Пусть $Z=aX+bY$. Возможные значения $z_k = a x_i + b y_j$ с вероятностями $p_{ij}=P(X=x_i,Y=y_j)$ (для независимости $p_{ij}=p_i q_j$). Тогда:
$$
\mathbb{E}[Z]=\sum_{i,j}(a x_i + b y_j)p_{ij}
= a\sum_i x_i p_i + b\sum_j y_j q_j
= a\mathbb{E}[X]+b\mathbb{E}[Y].
$$
В общем случае аналогично, только суммы идут по совместным распределениям.

### 3. Произведение независимых случайных величин
Если $X$ и $Y$ независимы, то:
$$
\mathbb{E}[XY] = \mathbb{E}[X]\cdot\mathbb{E}[Y]
$$
**Доказательство:**  
По определению для дискретного случая:
$$
\mathbb{E}[XY]=\sum_{i,j}x_i y_j P(X=x_i,Y=y_j)=\sum_{i,j}x_i y_j P(X=x_i)P(Y=y_j)
=(\sum_i x_i P(X=x_i))(\sum_j y_j P(Y=y_j))
=\mathbb{E}[X]\mathbb{E}[Y].
$$
Аналогично для непрерывного случая через двойной интеграл.

### 4. Неотрицательность
Если $X \geq 0$ почти наверное, то $\mathbb{E}[X] \geq 0$.

### 5. Монотонность
Если $X \leq Y$ п.н., то $\mathbb{E}[X] \leq \mathbb{E}[Y]$.

### 6. Условное математическое ожидание
$\mathbb{E}[X|Y]$ — это случайная величина, являющаяся функцией от $Y$, такая что:
$$
\mathbb{E}[X] = \mathbb{E}[\mathbb{E}[X|Y]]
$$
(Лемма о двойном ожидании).

---

## 3. Примеры вычисления

### Пример 1: Дискретная величина
Пусть $X$ принимает значения $-1, 0, 1$ с равными вероятностями $1/3$ каждое.
Тогда:
$$
\mathbb{E}[X] = (-1)\cdot\frac{1}{3} + 0\cdot\frac{1}{3} + 1\cdot\frac{1}{3} = -\frac{1}{3} + 0 + \frac{1}{3} = 0.
$$

### Пример 2: Непрерывная величина
Пусть $X$ имеет равномерное распределение на отрезке $[0,1]$, т.е. $f_X(x)=1$ при $x\in[0,1]$.
Тогда:
$$
\mathbb{E}[X] = \int_{0}^{1} x\cdot 1\,dx = \left[\frac{x^2}{2}\right]_0^1 = \frac{1}{2}.
$$

---

## literature

Вот подробный разбор главных действующих лиц рассказа И. С. Тургенева «Муму», а также анализ образа Герасима и идейного замысла автора.

### Главные действующие лица

**1. Герасим**
*   **Кто он:** Главный герой, дворник старой барыни (бабушки). Родом из деревни, глухонемой от рождения, но обладающий огромной физической силой и трудолюбием.
*   **Черты характера:**
    *   Трудолюбие и хозяйственность (он лучший работник в доме).
    *   Доброта и способность к глубокой привязанности (любовь к Татьяне, трепетная забота о Муму).
    *   Чувство собственного достоинства и внутренняя гордость.
    *   Молчаливое терпение, переходящее в протест только в критический момент.
*   **Роль в сюжете:** Центральный персонаж, вокруг которого строится повествование. Его трагедия раскрывает бесчеловечность крепостного права и вызывает у читателя сочувствие и осуждение жестокости помещиков.

**2. Барыня (Старая барыня)**
*   **Кто она:** Властная пожилая женщина, владелица дома, где служит Герасим.
*   **Черты характера:**
    *   Деспотизм и капризность.
    *   Эгоцентризм (мир вращается вокруг её желаний).
    *   Бессердечность и равнодушие к чувствам других людей.
    *   Непоследовательность решений.
*   **Роль в сюжете:** Антагонист Герасима. Именно её прихоть становится причиной гибели собаки и последующего ухода героя. Она олицетворяет всю систему крепостничества с её произволом.

**3. Татьяна**
*   **Кто она:** Прачка, работающая в том же доме, что и Герасим.
*   **Черты характера:**
    *   Терпеливость и покорность судьбе.
    *   Мягкость и безответность.
    *   Страх перед властью барыни.
*   **Роль в сюжете:** Первая любовь Герасима. Её судьба показывает, как легко ломаются судьбы простых людей под давлением господской воли (барыня насильно выдала её замуж за пьяницу Капитона).

**4. Муму**
*   **Кто она:** Маленькая собачка, спасённая Герасимом щенком.
*   **Черты характера:**
    *   Верность и преданность хозяину.
    *   Смелость (лаяла на чужих).
    *   Интуитивное понимание опасности (чувствовала неладное при появлении барыни).
*   **Роль в сюжете:** Символ любви и человечности Герасима. Гибель Муму становится последней каплей, переполняющей чашу терпения героя и толкающей его на бунт — уход обратно в деревню.

---

### Смысл образа Герасима

Герасим — это не просто несчастный глухонемой крестьянин. Это символ русского народа:
1.  **Сила духа:** Несмотря на физический недостаток и социальное положение раба, он обладает несокрушимым внутренним стержнем.
2.  **Связь с природой:** Он деревенский человек, которому тесно и душно в каменном городе. Природа для него естественна, а жизнь в услужении противоестественна.
3.  **Протест через молчание:** До определенного момента Герасим молча подчиняется, но его внутренний мир богат и глубок. Когда его чувства грубо растоптаны (гибель Муму), он находит способ выразить свой протест самым радикальным образом — уходом, разрывая цепи рабства.

Тургенев показал, что даже самый забитый и бесправный человек способен на чувство глубокого сострадания и имеет право на личное счастье. Убийство любимого существа по приказу разрушило душу Герасима, доказав, что человеческое достоинство нельзя подавить до конца.

### Какую идею хотел донести автор?

Главная идея рассказа — **критика крепостного права**. Тургенев наглядно демонстрирует, насколько эта система уродует души всех участников процесса:
*   Для крестьян она означает потерю свободы, разрушение семьи и невозможность личного счастья (судьба Татьяны, пьянство Капитона).
*   Для дворян она развращает, порождая самодурство, вседозволенность и атрофию человеческих чувств (характер барыни).

Финал рассказа трагичен, но закономерен: Герасим уходит в деревню, отказываясь быть частью этого мира насилия. Этим поступком Тургенев утверждает мысль о том, что истинная свобода возможна лишь там, где нет власти одного человека над другим.

---

# GigaChat-3-Lightning

## python

Вот простая и эффективная функция на Python, которая решает задачу подсчета слов с учетом регистра без учета знаков препинания:

### Решение
1. Используем `re` (регулярные выражения) для удаления пунктуации.
2. Приводим строку к нижнему регистру (`str.lower()`), чтобы регистр не учитывался.
3. Разбиваем строку по пробелам или символам-разделителям (`str.split()`).
4. Считаем количество вхождений каждого слова через словарь.

### Код функции

```python
import re
from collections import Counter

def word_count(text: str) -> dict:
    # Удаляем знаки препинания и приводим текст к нижнему регистру
    cleaned = re.sub(r'[^\w\s]', '', text).lower()
    
    # Делим на отдельные слова
    words = cleaned.split()
    
    # Подсчёт количества каждого слова
    return dict(Counter(words))
```

### Пример вызова и результат

```python
text = "Привет! Привет, как дела? Дела идут отлично!"
result = word_count(text)
print(result)
```

**Результат:**
```python
{'привет': 2, 'как': 1, 'дела': 2, 'идут': 1, 'отлично': 1}
```

- Регистр игнорируется: «Привет» и «привет» считаются одним словом.
- Знаки препинания удалены: «!» и «?» исчезают.
- Слово «дела» посчитано дважды.

---

## c

Вот полный компилируемый код на C, который динамически выделяет память под массив из $ n $ целых чисел, сортирует его пузырьковой сортировкой и корректно освобождает выделенную память:

**Код программы:**
```c
#include <stdio.h>
#include <stdlib.h> // Для malloc/free

// Функция пузырьковой сортировки
void bubble_sort(int *arr, int size) {
    for (int i = 0; i < size - 1; ++i) {
        // Каждый проход "выталкивает" максимальный элемент в конец массива
        for (int j = 0; j < size - i - 1; ++j) {
            if (arr[j] > arr[j + 1]) {
                // Меняем местами элементы
                int temp = arr[j];
                arr[j] = arr[j + 1];
                arr[j + 1] = temp;
            }
        }
    }
}

int main() {
    int n;
    
    // Вводим размер массива
    printf("Введите количество элементов (n): ");
    scanf("%d", &n);

    // Динамическое выделение памяти для массива
    int *arr = (int *)malloc(n * sizeof(int));
    if (!arr) {
        fprintf(stderr, "Ошибка выделения памяти!\n");
        return EXIT_FAILURE;
    }

    // Считывание значений с клавиатуры
    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; ++i) {
        scanf("%d", &arr[i]);
    }

    // Сортируем массив пузырьковой сортировкой
    bubble_sort(arr, n);

    // Вывод отсортированного массива
    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; ++i) {
        printf("%d ", arr[i]);
    }
    printf("\n");

    // Освобождение динамической памяти
    free(arr);

    return EXIT_SUCCESS;
}
```

---

### Как работает программа?

1. **Ввод данных:**  
   Сначала пользователь вводит число $ n $ — размер массива. Затем программа считывает $ n $ целых чисел.

2. **Динамическая память:**  
   Память выделяется через `malloc`, чтобы не ограничиваться фиксированным размером. Проверяется успешность выделения (`if (!arr)`).

3. **Пузырьковая сортировка:**  
   - Внешний цикл проходит по всем элементам массива ($ i $ от 0 до $ n-2 $).  
   - Внутренний цикл сравнивает соседние пары элементов и меняет их местами, если они стоят не по возрастанию.  
   - После каждого прохода наибольший элемент оказывается в конце массива.

4. **Освобождение памяти:**  
   После использования массива вызывается `free()` для предотвращения утечек памяти.

5. **Компиляция:**  
   Пример команды для компиляции и запуска:
   ```bash
   gcc program.c -o bubblesort
   ./bubblesort
   ```

Если нужно добавить проверку ввода или обработку ошибок (например, отрицательный размер), можно расширить программу, но базовая реализация полностью соответствует твоему запросу.

---

## math

**Математическое ожидание случайной величины**

Пусть $X$ — случайная величина, определённая на вероятностном пространстве.

---

### 1. Дискретный случай

Если $X$ принимает значения $x_1, x_2, \dots$ с соответствующими вероятностями $p_i = P(X=x_i)$ (где $\sum_{i=1}^{\infty} p_i = 1$), то математическим ожиданием называется:

$$ E[X] = \sum_{i=1}^{\infty} x_i p_i
$$

(Сумма конечна или сходится абсолютно).

---

### 2. Непрерывный случай

Если $X$ имеет плотность вероятности $f(x)$, то математическое ожидание определяется как:

$$ E[X] = \int_{-\infty}^{+\infty} x f(x)\,dx
$$

Интеграл должен быть сходящимся.

---

## Основные свойства математического ожидания

1. **Ожидание константы:**
   $$ E[c] = c, \quad c \in \mathbb{R}
   $$
   *Доказательство:* $E[c] = \int c f(x) dx = c\int f(x) dx = c$.

2. **Линейность:**
   - Для констант: $E[a + bX] = a + bE[X]$
   - Для суммы: $E[X+Y] = E[X] + E[Y]$ (при совместном существовании)
   - Для произведения константы и величины: $E[aX] = aE[X]$

   *Доказательства*:
   - $E[a+bX] = a\sum p_i + b\sum x_i p_i = a + bE[X]$
   - $E[X+Y] = \sum (x+y)p_{XY}(x,y) = \sum x p_X(x) + \sum y p_Y(y) = E[X]+E[Y]$

3. **Ожидание независимых величин:**
   Если $X$ и $Y$ независимы, $E[XY] = E[X]E[Y]$

   *Доказательство*:
   - $E[XY] = \sum_x \sum_y xy\,P(X=x,Y=y)$
   - По независимости: $xy\,P(X=x,Y=y)=xy\,P(X=x)P(Y=y)$
   - $E[XY] = \left(\sum_x x P(X=x)\right)\left(\sum_y y P(Y=y)\right) = E[X]E[Y]$

4. **Ожидание функции от случайной величины:**
   $E[g(X)] = \int g(x)f(x)dx$

5. **Неотрицательность (для неотрицательных):**
   Если $X \ge 0$, то $E[X] \ge 0$.

6. **Условное математическое ожидание:**
   $E[X|Y=y] = \int x f_{X|Y}(x|y) dx$

---

## Примеры вычисления

### Пример 1 (дискретная величина)

Случайная величина $X$ задана таблицей:

| $x$ | 1 | 2 | 3 |
|-----|---|---|---|
| $p$ | 0.2 | 0.5 | 0.3 |

Вычислим $E[X]$:

$$ E[X] = 1\cdot 0.2 + 2\cdot 0.5 + 3\cdot 0.3 = 0.2 + 1.0 + 0.9 = 2.1
$$

### Пример 2 (непрерывная величина)

Пусть $X \sim U(0, 1)$ (равномерно распределена на $[0,1]$). Плотность: $f(x) = 1$ для $x \in [0,1]$.

$$ E[X] = \int_0^1 x \cdot 1 \, dx = \frac{1}{2}x^2\Big|_0^1 = \frac{1}{2}
$$

---

**Итого:**  
- Определение: сумма/интеграл по вероятностям/плотности.
- Свойства: линейность, ожидание константы, произведение независимых, функция от величины.
- Примеры: дискретная ($X=1,2,3$ с заданными $p$) и непрерывная (равномерная на $[0,1]$).

---

## literature

Рассказ Ивана Сергеевича Тургенева «Муму» — это произведение, в котором ярко отражена тема крепостного права и бесправия простого народа. Главные действующие лица рассказа:

1. Герасим (главный герой)
- Кто он: дворник глухонемого крестьянина, крепостной мужик из деревни.
- Черты характера: сильный, трудолюбивый, добрый, честный, искренний, но замкнутый и молчаливый; обладает огромной физической силой и чувством собственного достоинства.
- Роль в сюжете: главный персонаж, вокруг которого строится повествование. Он олицетворяет силу русского крестьянства, его доброту и нравственную чистоту. Именно благодаря ему раскрывается жестокость и бездушие барского окружения.
- Смысл образа и идея автора: образ Герасима символизирует народ, который терпит унижения и несправедливость, сохраняя при этом внутреннюю духовную силу и человечность. Однако автор подчёркивает трагизм положения крепостных, вынужденных подчиняться произволу помещиков и лишённых свободы выбора своей судьбы. Трагическая гибель Муму становится символом не только личной драмы героя, но и всей трагедии бесправного народа.

2. Татьяна (прачка)
- Кто она: молодая прачка, вдова, живущая у барыни.
- Черты характера: робкая, скромная, тихая, покорная судьбе, с добрым сердцем.
- Роль в сюжете: её судьба связана с судьбой Герасима. Сначала барыне нравится Герасим, и она пытается заигрывать с ним, однако тот отвергает ухаживания, испытывая к Татьяне жалость и уважение. Позже Татьяна выходит замуж за пьяницу Капитона, что вызывает грусть и сожаление у читателя.
- Значение образа: символ забитой, бесправной женщины-крепостной, чья жизнь полностью зависит от воли хозяйки.

3. Барыня (помещица)
- Кто она: властная, капризная, деспотичная старуха-помещица.
- Черты характера: высокомерная, жестока, капризна, равнодушна к страданиям окружающих людей.
- Роль в сюжете: является центральной фигурой, определяющей судьбу героев. Она решает судьбу Татьяны, заставляет Герасима жениться на прачке, а затем приказывает избавиться от собаки Муму.
- Значение образа: воплощение социального зла, жестокости и произвола господствующего класса, бессердечия и самодурства.

4. Капитон Климов (муж Татьяны)
- Кто он: пьянствующий сапожник, муж Татьяны.
- Черты характера: слабовольный, ленивый, неспособен взять ответственность за свою семью.
- Роль в сюжете: после неудачной попытки спиться и сбежать из дома барыни Капитон окончательно опускается до брака с Татьяной.
- Значение образа: пример человека, сломленного обстоятельствами и собственным характером.

5. Муму (собака)
- Кто она: маленькая собачка, которую Герасим спасает и привязывается к ней всем сердцем.
- Черты характера: преданная, ласковая, умная, доверчивая.
- Роль в сюжете: символ чистоты и искренности души, единственный друг и радость для Герасима. Её смерть становится высшей точкой страдания главного героя.
- Значение образа: символ утраченной надежды, любви и счастья, которые были отняты у крепостного человека.

Общий смысл произведения заключается в том, чтобы показать бесчеловечность крепостнической системы, где человек воспринимается как вещь, а личность подавляется властью. Через трагедию Герасима Тургенев выражает идею о необходимости уважения человеческого достоинства и протестует против социальной несправедливости.

---

# Работа с DeepSeek через API

Работу с DeepSeek будем проводить через SDK OpenAI (API имеет такой же формат)

## Получение API ключа

[DeepSeek](https://platform.deepseek.com) не предоставляет бесплатных токенов, а так же нет возможности пополнить аккаунт с Российских карт. 

Несколько миллионов токенов были купленны на стронней площадке. С ключем идет `base-url`, к которому следует обращатся, для использования API.

## Получение списка доступных моделей

Так как токены были купленны у агрегатора моделей, в списке будут и другие модели. Отобразим только модели от DeepSeek.

In [20]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://ai.starimg.ru/v1",
)

for m in client.models.list().data:
    if "deepseek" in m.id:
        print(m.id)

deepseek-v4-pro
deepseek-v4.1-flash
deepseek-v4-flash


## Отправка запросов на API

In [21]:
resp = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[{"role": "user", "content": "Привет, как дела?"}],
)

answer = resp.choices[0].message.content
print(answer)

Привет! У меня всё отлично — готов помочь с чем угодно. А как у тебя дела?


При помощи роли `system` можно скорректировать поведение модели

In [22]:
resp = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[
        {"role": "system", "content": "Отвечай одним предложением, в конце добавляй смайлик"},
        {"role": "user", "content": "Привет, как дела?"},
    ],
)

answer = resp.choices[0].message.content
print(answer)

Привет, у меня всё отлично, спасибо, а как твои дела? 😊


### Отправим несколько запросов моделям DeepSeek-v4-pro и DeepSeek-v4.1-flash

In [23]:
from IPython.display import Markdown, display

models = ["deepseek-v4-pro", "deepseek-v4.1-flash"]

for model in models:
    display(Markdown(f"# {model}"))
    for prompt in prompts:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt["prompt"]}],
        )
        answer = resp.choices[0].message.content
        display(Markdown(f"## {prompt["name"]}\n\n{answer}\n\n---"))

# deepseek-v4-pro

## python

```python
import re
from collections import Counter

def count_words(text: str) -> dict:
    """
    Возвращает словарь, в котором ключи — слова, а значения — количество их вхождений.
    Регистр не учитывается, знаки препинания игнорируются.
    """
    # Приводим строку к нижнему регистру
    text = text.lower()
    # Удаляем все символы, которые не являются буквами, цифрами или пробелами
    cleaned_text = re.sub(r'[^\w\s]', '', text)
    # Разбиваем оставшийся текст на слова по пробельным символам
    words = cleaned_text.split()
    # Подсчитываем частоты с помощью Counter и возвращаем как обычный словарь
    return dict(Counter(words))

# Пример вызова
example = "Hello, world! Hello, Python. Python is great, isn't it?"
result = count_words(example)
print(result)
```

**Результат выполнения:**
```
{'hello': 2, 'world': 1, 'python': 2, 'is': 1, 'great': 1, 'isnt': 1, 'it': 1}
```

### Краткое объяснение
1. **Приведение к нижнему регистру** – чтобы слова `"Hello"` и `"hello"` считались одинаковыми.
2. **Удаление знаков препинания** – с помощью регулярного выражения `[^\w\s]` удаляем все символы, не являющиеся буквами, цифрами или пробелами (знаки препинания, включая апострофы и дефисы, исчезают, поэтому `"isn't"` превращается в `"isnt"`).
3. **Разбиение на слова** – метод `split()` делит строку по любым пробельным символам (пробел, табуляция, перенос строки).
4. **Подсчёт частот** – `collections.Counter` автоматически считает количество вхождений каждого элемента списка.

---

## c

```c
#include <stdio.h>
#include <stdlib.h>

// Функция пузырьковой сортировки (оптимизированная версия с флагом)
void bubble_sort(int *arr, int n) {
    int swapped;
    for (int i = 0; i < n - 1; i++) {
        swapped = 0;
        for (int j = 0; j < n - i - 1; j++) {
            if (arr[j] > arr[j + 1]) {
                // Обмен элементов
                int temp = arr[j];
                arr[j] = arr[j + 1];
                arr[j + 1] = temp;
                swapped = 1;
            }
        }
        // Если за проход не было обменов, массив уже отсортирован
        if (!swapped) {
            break;
        }
    }
}

int main(void) {
    int n;

    // 1. Ввод количества элементов
    printf("Введите количество элементов: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        printf("Ошибка: необходимо ввести положительное целое число.\n");
        return 1;
    }

    // 2. Динамическое выделение памяти
    int *arr = (int*)malloc(n * sizeof(int));
    if (arr == NULL) {
        printf("Ошибка: не удалось выделить память.\n");
        return 1;
    }

    // 3. Ввод элементов массива
    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; i++) {
        if (scanf("%d", &arr[i]) != 1) {
            printf("Ошибка ввода элемента %d.\n", i + 1);
            free(arr);
            return 1;
        }
    }

    // 4. Сортировка пузырьком
    bubble_sort(arr, n);

    // 5. Вывод результата
    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; i++) {
        printf("%d ", arr[i]);
    }
    printf("\n");

    // 6. Освобождение памяти
    free(arr);

    return 0;
}
```

### Пояснение работы программы

1. **Ввод количества элементов**  
   С помощью `scanf` считывается целое число `n`. Если ввод некорректен или `n <= 0`, программа завершается с сообщением об ошибке.

2. **Динамическое выделение памяти**  
   Функция `malloc` выделяет блок памяти под `n` целых чисел. Если выделение не удалось (возвращён `NULL`), программа сообщает об ошибке и завершается.

3. **Ввод массива**  
   В цикле считываются `n` целых чисел и сохраняются в выделенную память. При ошибке ввода память освобождается, и программа завершается.

4. **Пузырьковая сортировка**  
   Функция `bubble_sort` выполняет сортировку массива по возрастанию. Внешний цикл проходит `n-1` раз, внутренний — попарно сравнивает соседние элементы и меняет их местами, если они стоят в неправильном порядке. Переменная `swapped` используется для раннего выхода: если за полный проход не было ни одного обмена, массив уже отсортирован.

5. **Вывод отсортированного массива**  
   Все элементы выводятся через пробел в одной строке.

6. **Освобождение памяти**  
   Вызов `free(arr)` возвращает выделенную память системе.

Программа компилируется любым компилятором C (например, `gcc -std=c99 -Wall program.c -o program`) и корректно работает с динамической памятью.

---

## math

Пусть \((\Omega,\mathcal F,\mathbb P)\) — вероятностное пространство.  
Общее строгое определение такое:

\[
\mathbb E X=\int_\Omega X(\omega)\,d\mathbb P(\omega),
\]

если интеграл Лебега существует. Для конечного математического ожидания требуется \(\mathbb E|X|<\infty\).

---

## 1. Дискретный случай

Случайная величина \(X\) дискретна, если она принимает не более чем счётное множество значений \(x_1,x_2,\dots\) с вероятностями

\[
p_i=\mathbb P(X=x_i).
\]

Тогда

\[
\boxed{\mathbb E X=\sum_i x_i p_i }
\]

если ряд сходится абсолютно:

\[
\sum_i |x_i|p_i<\infty.
\]

Если абсолютная сходимость отсутствует, математическое ожидание не определено.

---

## 2. Непрерывный случай

Случайная величина \(X\) называется абсолютно непрерывной, если существует плотность \(f_X(x)\ge 0\), такая что для любого борелевского множества \(B\)

\[
\mathbb P(X\in B)=\int_B f_X(x)\,dx.
\]

Тогда математическое ожидание равно

\[
\boxed{\mathbb E X=\int_{-\infty}^{\infty} x f_X(x)\,dx }
\]

если

\[
\int_{-\infty}^{\infty} |x| f_X(x)\,dx<\infty.
\]

---

# Основные свойства

Будем предполагать, что указанные математические ожидания существуют.

### 1. Математическое ожидание константы

Если \(X=c\) почти наверное, то

\[
\mathbb E X=c.
\]

**Доказательство.**  
Величина \(X\) дискретна, принимает единственное значение \(c\) с вероятностью \(1\), поэтому

\[
\mathbb E X=c\cdot \mathbb P(X=c)=c\cdot 1=c.
\]

---

### 2. Линейность

Для любых случайных величин \(X,Y\) и чисел \(a,b\)

\[
\mathbb E(aX+bY)=a\mathbb E X+b\mathbb E Y.
\]

**Доказательство в дискретном случае.**  
Пусть \(X\) принимает значения \(x_i\), \(Y\) — значения \(y_j\). Обозначим

\[
p_{ij}=\mathbb P(X=x_i,Y=y_j).
\]

Тогда

\[
\begin{aligned}
\mathbb E(aX+bY)
&=\sum_{i,j}(a x_i+b y_j)p_{ij}\\
&=a\sum_i x_i\sum_j p_{ij}
+b\sum_j y_j\sum_i p_{ij}\\
&=a\sum_i x_i\mathbb P(X=x_i)
+b\sum_j y_j\mathbb P(Y=y_j)\\
&=a\mathbb E X+b\mathbb E Y.
\end{aligned}
\]

**Доказательство в непрерывном случае.**  
Если \((X,Y)\) имеет совместную плотность \(f_{X,Y}(x,y)\), то

\[
\begin{aligned}
\mathbb E(aX+bY)
&=\iint (a x+b y)f_{X,Y}(x,y)\,dx\,dy\\
&=a\int x\left(\int f_{X,Y}(x,y)\,dy\right)dx
+b\int y\left(\int f_{X,Y}(x,y)\,dx\right)dy\\
&=a\int x f_X(x)\,dx+b\int y f_Y(y)\,dy\\
&=a\mathbb E X+b\mathbb E Y.
\end{aligned}
\]

---

### 3. Неотрицательность и монотонность

Если \(X\ge 0\) почти наверное, то

\[
\mathbb E X\ge 0.
\]

Если \(X\le Y\) почти наверное, то

\[
\mathbb E X\le \mathbb E Y.
\]

**Доказательство.**  
Для неотрицательной дискретной величины все \(x_i\ge 0\) с положительной вероятностью, поэтому сумма неотрицательна. Для непрерывной величины плотность \(f_X(x)=0\) при \(x<0\), поэтому интеграл от \(x f_X(x)\) неотрицателен.

Если \(X\le Y\), то \(Y-X\ge 0\). Тогда

\[
\mathbb E(Y-X)\ge 0.
\]

По линейности

\[
\mathbb E Y-\mathbb E X\ge 0,
\]

то есть

\[
\mathbb E X\le \mathbb E Y.
\]

---

### 4. Математическое ожидание функции от случайной величины

Если \(g:\mathbb R\to\mathbb R\) — борелевская функция, то

\[
\mathbb E g(X)=
\begin{cases}
\sum_i g(x_i)\mathbb P(X=x_i), & X\text{ дискретна},\\[4pt]
\displaystyle\int_{-\infty}^{\infty} g(x)f_X(x)\,dx, & X\text{ непрерывна},
\end{cases}
\]

при условии абсолютной сходимости суммы или интеграла.

Это свойство часто называют «правилом бессознательного статистика».

---

### 5. Математическое ожидание произведения независимых величин

Если \(X\) и \(Y\) независимы и \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то

\[
\mathbb E(XY)=\mathbb E X\,\mathbb E Y.
\]

**Доказательство в дискретном случае.**  
Пусть \(X\) принимает значения \(x_i\), а \(Y\) — значения \(y_j\). По независимости

\[
\mathbb P(X=x_i,Y=y_j)=\mathbb P(X=x_i)\mathbb P(Y=y_j).
\]

Тогда

\[
\begin{aligned}
\mathbb E(XY)
&=\sum_{i,j}x_i y_j\mathbb P(X=x_i,Y=y_j)\\
&=\sum_{i,j}x_i y_j\mathbb P(X=x_i)\mathbb P(Y=y_j)\\
&=\left(\sum_i x_i\mathbb P(X=x_i)\right)
\left(\sum_j y_j\mathbb P(Y=y_j)\right)\\
&=\mathbb E X\,\mathbb E Y.
\end{aligned}
\]

**Доказательство в непрерывном случае.**  
Если \(X\) и \(Y\) независимы и имеют плотности \(f_X,f_Y\), то совместная плотность равна

\[
f_{X,Y}(x,y)=f_X(x)f_Y(y).
\]

Следовательно,

\[
\begin{aligned}
\mathbb E(XY)
&=\iint xy f_X(x)f_Y(y)\,dx\,dy\\
&=\left(\int x f_X(x)\,dx\right)
\left(\int y f_Y(y)\,dy\right)\\
&=\mathbb E X\,\mathbb E Y.
\end{aligned}
\]

---

### 6. Формула для дисперсии

Если \(\mathbb E X^2<\infty\), то

\[
\operatorname{Var}X=\mathbb E[(X-\mathbb E X)^2]
=\mathbb E X^2-(\mathbb E X)^2.
\]

**Доказательство.**  
Обозначим \(\mu=\mathbb E X\). Тогда

\[
\begin{aligned}
\mathbb E[(X-\mu)^2]
&=\mathbb E[X^2-2\mu X+\mu^2]\\
&=\mathbb E X^2-2\mu\mathbb E X+\mu^2\\
&=\mathbb E X^2-2\mu^2+\mu^2\\
&=\mathbb E X^2-\mu^2.
\end{aligned}
\]

---

# Примеры

## Дискретный пример

Пусть \(X\) — число очков при броске правильной шестигранной кости. Тогда

\[
\mathbb P(X=k)=\frac16,\qquad k=1,2,\dots,6.
\]

Следовательно,

\[
\mathbb E X
=\sum_{k=1}^6 k\cdot \frac16
=\frac{1+2+3+4+5+6}{6}
=\frac{21}{6}
=\frac72
=3{,}5.
\]

---

## Непрерывный пример

Пусть \(X\sim \mathrm{Exp}(\lambda)\), \(\lambda>0\). Плотность:

\[
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x}, & x\ge 0,\\
0, & x<0.
\end{cases}
\]

Тогда

\[
\mathbb E X
=\int_0^\infty x\lambda e^{-\lambda x}\,dx.
\]

Сделаем замену \(u=\lambda x\), тогда \(dx=\frac{du}{\lambda}\). Получаем

\[
\mathbb E X
=\frac1\lambda\int_0^\infty u e^{-u}\,du.
\]

Интеграл

\[
\int_0^\infty u e^{-u}\,du=1,
\]

поэтому

\[
\mathbb E X=\frac1\lambda.
\]

Например, при \(\lambda=1\) имеем \(\mathbb E X=1\).

---

## literature

## Главные действующие лица рассказа И. С. Тургенева «Муму»

### Герасим

**Кто он:** глухонемой дворник, крепостной крестьянин, привезённый из деревни в московский дом старой барыни. Мужчина огромного роста и необыкновенной физической силы.

**Черты характера:** трудолюбивый, добрый, честный, верный, способный на глубокую и преданную привязанность. Несмотря на физический недостаток, он наделён душевной чуткостью и чувством собственного достоинства. В нём сочетаются богатырская мощь и нравственная чистота, но вместе с тем — привычка к беспрекословному повиновению.

**Роль в сюжете:** центральный герой. Его жизнь, любовь к прачке Татьяне, привязанность к собачке Муму и трагическое подчинение барскому приказу раскрывают главный конфликт рассказа — столкновение живого человеческого чувства с жестокой крепостнической системой.

### Барыня

**Кто она:** старая помещица-вдова, владелица многочисленных крепостных. Живёт в Москве, окружённая прислугой и приживалками.

**Черты характера:** капризная, своенравная, деспотичная, равнодушная к судьбам своих крепостных. Её жизнь пуста и скучна, поэтому она развлекается, распоряжаясь людьми, не задумываясь о последствиях своих прихотей. Она не столько зла, сколько привыкла к полной власти над людьми.

**Роль в сюжете:** олицетворяет помещичий произвол. Её капризы становятся движущей силой трагических событий: сначала она выдаёт Татьяну замуж за пьяницу Капитона, затем требует избавиться от Муму. Именно её воля разрушает счастье Герасима.

### Татьяна

**Кто она:** крепостная прачка, женщина лет двадцати восьми.

**Черты характера:** тихая, робкая, запуганная, безропотная. Всю жизнь она привыкла подчиняться и не имеет сил сопротивляться. Она трудолюбива, но её душевная подавленность не позволяет ей даже ответить на любовь Герасима.

**Роль в сюжете:** объект первой привязанности Герасима. Её насильственная выдача замуж за Капитона показывает полное бесправие крепостных людей и становится для Герасима первым тяжёлым ударом.

### Капитон Климов

**Кто он:** башмачник, крепостной, горький пьяница.

**Черты характера:** никчёмный, ленивый, слабохарактерный, но при этом с претензией на «образованность» и «тонкость чувств». Он не злой, но жалкий человек, сломленный крепостной действительностью.

**Роль в сюжете:** его женитьба на Татьяне, устроенная по приказу барыни, окончательно лишает Герасима надежды на личное счастье. Капитон — ещё одна жертва крепостного права, хотя видимой причиной бед становится именно он.

### Гаврила

**Кто он:** дворецкий в доме барыни.

**Черты характера:** исполнительный, трусливый, угодливый. Он не имеет собственной воли и лишь следит за тем, чтобы приказы барыни выполнялись точно и без возражений.

**Роль в сюжете:** посредник между барыней и остальными крепостными. Именно через него передаётся приказ об уничтожении Муму.

### Муму

**Кто она:** маленькая собачка, спасённая Герасимом из реки и выкормленная им.

**Черты характера:** верная, ласковая, беззащитная, преданная хозяину.

**Роль в сюжете:** становится для Герасима тем единственным существом, которое бескорыстно любит его. Её гибель — кульминация рассказа, момент, когда герой теряет всё, что ему дорого. Муму — символ живого чувства, безжалостно уничтоженного барской волей.

---

## Смысл образа Герасима и идея автора

Образ Герасима — один из самых значимых в русской литературе. Это **олицетворение русского народа**: внешне могучий, трудолюбивый, нравственно здоровый, но в условиях крепостного права — бесправный и немой. Его глухонемота — не только физический недостаток, но и **символ подневольного молчания**, невозможности открыто выразить протест. Он не может говорить, но чувствует глубоко и сильно.

Герасим наделён чертами былинного богатыря, однако его сила не находит достойного применения: он вынужден подчиняться прихотям капризной старухи. Любовь к Татьяне и привязанность к Муму показывают, что он способен на искренние, глубокие чувства. Однако система, в которой он живёт, отнимает у него сначала любимую женщину, а затем и единственное живое существо, отвечавшее ему любовью.

Убийство Муму — трагический перелом. Герасим впервые проявляет самостоятельность: он подчиняется приказу, но затем **уходит от барыни обратно в деревню**. Этот шаг — стихийный, молчаливый, но мощный протест против произвола. Он не может открыто восстать, но он больше не желает оставаться рабом.

Тургенев показывает, что даже самый покорный человек имеет предел терпения. Писатель осуждает крепостное право, которое калечит судьбы, лишает людей свободы, любви и человеческого достоинства. Вместе с тем в образе Герасима автор выражает веру в скрытую силу народа, которая рано или поздно должна проснуться. Идея рассказа — протест против угнетения человека человеком и утверждение права каждой личности на чувство, свободу и внутреннюю независимость.

---

# deepseek-v4.1-flash

## python

```python
import re
from collections import Counter

def word_count(text: str) -> dict[str, int]:
    words = re.findall(r'[^\W_]+', text.lower(), flags=re.UNICODE)
    return dict(Counter(words))
```

**Кратко как работает:**
- `text.lower()` — приводит строку к нижнему регистру.
- `re.findall(r'[^\W_]+', ...)` — находит последовательности букв и цифр, игнорируя знаки препинания и подчёркивания.
- `Counter(words)` — подсчитывает количество каждого слова.
- `dict(...)` — возвращает обычный словарь.

**Пример вызова:**

```python
text = "Привет, мир! Привет, Python. Python — это мир."
print(word_count(text))
```

**Результат:**

```python
{'привет': 2, 'мир': 2, 'python': 2, 'это': 1}
```

---

## c

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int n;
    printf("Введите количество элементов: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        fprintf(stderr, "Некорректное значение n\n");
        return 1;
    }

    int *a = malloc((size_t)n * sizeof(int));
    if (a == NULL) {
        fprintf(stderr, "Ошибка выделения памяти\n");
        return 1;
    }

    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; i++) {
        if (scanf("%d", &a[i]) != 1) {
            fprintf(stderr, "Ошибка ввода\n");
            free(a);
            return 1;
        }
    }

    /* Пузырьковая сортировка */
    for (int i = 0; i < n - 1; i++) {
        int swapped = 0;
        for (int j = 0; j < n - 1 - i; j++) {
            if (a[j] > a[j + 1]) {
                int tmp = a[j];
                a[j] = a[j + 1];
                a[j + 1] = tmp;
                swapped = 1;
            }
        }
        if (!swapped) break; /* массив уже отсортирован */
    }

    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; i++) {
        printf("%d ", a[i]);
    }
    printf("\n");

    free(a);
    return 0;
}
```

### Как работает программа

1. **Ввод `n`.** Считываем количество элементов и проверяем, что оно положительное.
2. **Выделение памяти.** Через `malloc` выделяем `n * sizeof(int)` байт под массив. Проверяем, что память выделилась (`a != NULL`).
3. **Чтение массива.** В цикле считываем `n` целых чисел в массив. При ошибке ввода освобождаем память и завершаем работу.
4. **Пузырьковая сортировка.**
   - Внешний цикл `i` — номер прохода. После каждого прохода наибольший из ещё не отсортированных элементов «всплывает» в конец.
   - Внутренний цикл `j` идёт до `n - 1 - i`, потому что последние `i` элементов уже стоят на своих местах.
   - Если соседние элементы стоят в неправильном порядке (`a[j] > a[j+1]`), меняем их местами и ставим флаг `swapped = 1`.
   - Если за проход не было ни одной перестановки — массив уже отсортирован, выходим из внешнего цикла (оптимизация).
5. **Вывод результата.** Печатаем отсортированный массив.
6. **Освобождение памяти.** Вызываем `free(a)`, чтобы не было утечки.

Сложность алгоритма: O(n²) в худшем случае, O(n) — в лучшем (уже отсортированный массив).

---

## math

## 1. Строгое определение

Пусть задано вероятностное пространство \((\Omega,\mathcal F,\mathbb P)\), а \(X:\Omega\to\mathbb R\) — случайная величина. В общем случае математическое ожидание определяется как интеграл Лебега
\[
\mathbb E X=\int_\Omega X(\omega)\,d\mathbb P(\omega),
\]
если \(\mathbb E|X|=\int_\Omega |X|\,d\mathbb P<\infty\). Если интеграл не абсолютно сходится, математическое ожидание в стандартном смысле не определено (для неотрицательных величин допускается значение \(+\infty\)).

### Дискретный случай

Пусть \(X\) принимает не более чем счётное множество значений \(x_1,x_2,\dots\) с вероятностями
\[
p_i=\mathbb P(X=x_i),\qquad \sum_i p_i=1.
\]
Тогда
\[
\boxed{\mathbb E X=\sum_i x_i p_i}
\]
при условии
\[
\sum_i |x_i|p_i<\infty.
\]
Если \(X\ge 0\), сумма может быть и бесконечной.

### Непрерывный случай

Пусть \(X\) — абсолютно непрерывная случайная величина с плотностью \(f_X(x)\). Тогда
\[
\boxed{\mathbb E X=\int_{-\infty}^{\infty} x f_X(x)\,dx}
\]
при условии
\[
\int_{-\infty}^{\infty} |x|f_X(x)\,dx<\infty.
\]
Если \(X\ge 0\), интеграл может быть равен \(+\infty\).

---

## 2. Основные свойства математического ожидания

### 1) Ожидание константы
Если \(X\equiv c\) почти наверное, то
\[
\mathbb E[c]=c.
\]
Доказательство:
\[
\mathbb E[c]=\int_\Omega c\,d\mathbb P=c\mathbb P(\Omega)=c.
\]

### 2) Линейность
Если \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то для любых \(a,b\in\mathbb R\)
\[
\mathbb E[aX+bY]=a\mathbb E X+b\mathbb E Y.
\]
Доказательство: по линейности интеграла Лебега
\[
\mathbb E[aX+bY]=\int_\Omega (aX+bY)\,d\mathbb P
=a\int_\Omega X\,d\mathbb P+b\int_\Omega Y\,d\mathbb P
=a\mathbb E X+b\mathbb E Y.
\]
В дискретном случае это сумма по совместному распределению, в непрерывном — двойной интеграл по совместной плотности. Независимость здесь не нужна.

### 3) Монотонность
Если \(X\le Y\) почти наверное, то
\[
\mathbb E X\le \mathbb E Y.
\]
Доказательство: \(Y-X\ge 0\) почти наверное, а математическое ожидание неотрицательной величины неотрицательно:
\[
\mathbb E Y-\mathbb E X=\mathbb E[Y-X]\ge 0.
\]

### 4) Ожидание индикатора
Для события \(A\in\mathcal F\)
\[
\mathbb E[\mathbf 1_A]=\mathbb P(A).
\]
Доказательство:
\[
\mathbb E[\mathbf 1_A]=\int_\Omega \mathbf 1_A\,d\mathbb P=\mathbb P(A).
\]

### 5) Ожидание произведения независимых величин
Если \(X\) и \(Y\) независимы и \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то
\[
\mathbb E[XY]=\mathbb E X\cdot \mathbb E Y.
\]
Доказательство в дискретном случае. Пусть
\[
p_i=\mathbb P(X=x_i),\qquad q_j=\mathbb P(Y=y_j).
\]
Тогда в силу независимости
\[
\mathbb P(X=x_i,Y=y_j)=p_iq_j.
\]
Следовательно,
\[
\mathbb E[XY]=\sum_{i,j}x_iy_jp_iq_j
=\left(\sum_i x_ip_i\right)\left(\sum_j y_jq_j\right)
=\mathbb E X\cdot \mathbb E Y.
\]
В непрерывном случае, если \(f_{X,Y}(x,y)=f_X(x)f_Y(y)\),
\[
\mathbb E[XY]=\int\!\!\int xy\,f_X(x)f_Y(y)\,dx\,dy
=\left(\int x f_X(x)\,dx\right)\left(\int y f_Y(y)\,dy\right).
\]

### 6) Ожидание функции от случайной величины
В дискретном случае
\[
\mathbb E[g(X)]=\sum_i g(x_i)p_i,
\]
в непрерывном случае
\[
\mathbb E[g(X)]=\int_{-\infty}^{\infty} g(x)f_X(x)\,dx,
\]
если соответствующий ряд или интеграл сходится абсолютно.

---

## 3. Примеры вычисления

### Дискретный пример: честный кубик

Пусть \(X\) — число выпавших очков на честном игральном кубике. Тогда
\[
\mathbb P(X=k)=\frac16,\qquad k=1,2,\dots,6.
\]
Следовательно,
\[
\mathbb E X=\sum_{k=1}^6 k\cdot \frac16
=\frac{1+2+3+4+5+6}{6}
=\frac{21}{6}=3{,}5.
\]

### Непрерывный пример: показательное распределение

Пусть \(X\sim \mathrm{Exp}(\lambda)\), \(\lambda>0\). Её плотность:
\[
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x}, & x\ge 0,\\
0, & x<0.
\end{cases}
\]
Тогда
\[
\mathbb E X=\int_0^\infty x\lambda e^{-\lambda x}\,dx.
\]
Интегрируя по частям или заменой \(u=\lambda x\),
\[
\mathbb E X=\frac1\lambda.
\]
Например, при \(\lambda=2\) имеем \(\mathbb E X=0{,}5\).

---

## literature

«Муму» — рассказ о крепостной неволе, в центре которого трагическая судьба глухонемого дворника Герасима. Главные действующие лица — Герасим, Муму, барыня, Татьяна, Капитон и дворецкий Гаврила.

**Герасим**  
Кто он: крепостной крестьянин, глухонемой от рождения, огромной физической силы, привезённый барыней из деревни в Москву и назначенный дворником.  
Черты характера: трудолюбивый, честный, суровый внешне, но добрый и чуткий; замкнутый, одинокий, терпеливый, способный на глубокую любовь и привязанность. При этом в нём есть чувство собственного достоинства.  
Роль в сюжете: центральный герой. Его судьба организует всё повествование. Любовь к Татьяне и привязанность к Муму — единственные светлые чувства в его жизни. По прихоти барыни он теряет Татьяну, а потом вынужден убить Муму. После этого уходит из Москвы в деревню. Это кульминация и развязка рассказа.

**Муму**  
Кто она: маленькая собачка, спасённая Герасимом из реки.  
Черты: преданная, ласковая, беззащитная, привязанная только к Герасиму.  
Роль в сюжете: символ единственного тепла и любви в жизни Герасима. Её появление даёт ему счастье, а конфликт с барыней и гибель становятся страшным обвинением крепостничеству. Муму — невинная жертва барского самодурства.

**Барыня**  
Кто она: старая московская помещица, владелица Герасима и других крепостных.  
Черты: деспотичная, капризная, эгоистичная, мнительная, жестокая. Её чувства — это прихоти, она не считается с живыми людьми.  
Роль в сюжете: главная антагонистка. Именно её произвол ломает судьбы Татьяны, Капитона и Герасима, губит Муму. Она олицетворяет бесконтрольную власть крепостного права.

**Татьяна**  
Кто она: молодая прачка, крепостная барыни.  
Черты: тихая, кроткая, трудолюбивая, запуганная, покорная. Она привыкла терпеть и не смеет сопротивляться.  
Роль в сюжете: объект любви Герасима. По приказу барыни её выдают замуж за Капитона, и это становится для Герасима тяжёлым ударом. Её судьба показывает полное бесправие крепостных в личной жизни.

**Капитон Климов**  
Кто он: сапожник, крепостной, горький пьяница.  
Черты: слабохарактерный, распущенный, трусливый, жалкий; пытается казаться образованным.  
Роль в сюжете: муж Татьяны. Его женитьба на ней разлучает её с Герасимом. Пьянство и деградация Капитона — следствие жизни в крепостной неволе, хотя и сам он изображён неприятно.

**Гаврила Андреевич**  
Кто он: дворецкий, главный управляющий дворни барыни.  
Черты: хитрый, услужливый, боязливый перед барыней, готовый выполнить любой её приказ. Сам крепостной, но угнетает других.  
Роль в сюжете: посредник между барыней и дворней. Именно он передаёт Герасиму приказ избавиться от Муму. Через этот образ показано, как крепостническая система развращает и самих слуг.

**Смысл образа Герасима и идея автора**

Герасим — воплощение лучших черт русского народа: богатырской силы, трудолюбия, честности, доброты и способности к глубокой любви. Но он глухонемой и крепостной. Его немота символична: он не может сказать «нет», не может открыто протестовать, его сила и чувства принадлежат барыне. Он терпит, пока хватает сил: теряет Татьяну, потом Муму. Убийство Муму он совершает сам, потому что не смеет ослушаться, но после этого его терпение кончается. Уход в деревню — это немой бунт, нравственная победа и отказ быть рабом.

Тургенев хотел показать, что крепостное право калечит и губит людей, разрывает естественные человеческие связи, превращает одних в жертв, других — в палачей и прислужников. Автор осуждает деспотизм и безнаказанную власть помещиков, но вместе с тем сочувствует простому народу и верит в его нравственную силу. Образ Герасима — это не только обвинение крепостничеству, но и утверждение человеческого достоинства, которое нельзя уничтожить до конца.

---

# Краткие итоги

GigaChat и DeepSeek оба предоставляют удобное и очень схожее API.  

У DeepSeek покупка токенов в РФ менее удобна.